<a href="https://colab.research.google.com/github/PandoraRiot/Clasificaciondefrutas.udea.novateam/blob/dev/clasificacionfrutascolab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os

# Esta lógica detecta el entorno automáticamente
if 'google.colab' in str(get_ipython()):
    print("Ejecutando en Colab: Clonando repositorio...")
    repo_path = "/content/Clasificaciondefrutas.udea.novateam"
    if not os.path.exists(repo_path):
        !git clone https://github.com/PandoraRiot/Clasificaciondefrutas.udea.novateam.git
    os.chdir(repo_path)
    base_dir = "data/selected"
else:
    print("Ejecutando en VSCode: Usando rutas locales...")
    base_dir = "../data/selected"

# Verificación de carpetas
train_dir = os.path.join(base_dir, "train")
if not os.path.exists(train_dir):
    train_dir = os.path.join(base_dir, "Training")

Ejecutando en Colab: Clonando repositorio...


In [3]:
# ========================================================
# 1. PREPARACIÓN DEL ENTORNO (Repositorio y Librerías)
# ========================================================
import os

# Clonar el repositorio si no existe
if not os.path.exists('Clasificaciondefrutas.udea.novateam'):
    !git clone https://github.com/PandoraRiot/Clasificaciondefrutas.udea.novateam.git

%cd Clasificaciondefrutas.udea.novateam
!pip install -r requirements.txt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix



Cloning into 'Clasificaciondefrutas.udea.novateam'...
remote: Enumerating objects: 6596, done.
remote: Counting objects: 100% (6596/6596), done.
remote: Compressing objects: 100% (6587/6587), done.
remote: Total 6596 (delta 9), reused 6584 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (6596/6596), 27.98 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (6535/6535), done.
/content/Clasificaciondefrutas.udea.novateam/Clasificaciondefrutas.udea.novateam/Clasificaciondefrutas.udea.novateam


In [4]:
# ========================================================
# 2. CONFIGURACIÓN DE RUTAS DINÁMICAS
# ========================================================
# Definimos las rutas posibles para Colab o Local
posibles_rutas = [
    "/content/Clasificaciondefrutas.udea.novateam/data/selected",
    "data/selected",
    "../data/selected"
]

base_dir = None
for ruta in posibles_rutas:
    # Verificamos si existe la carpeta o alguna subcarpeta clave
    if os.path.exists(ruta):
        base_dir = ruta
        break

if base_dir:
    # Detectar si las carpetas están en minúsculas (train/test) o Mayúsculas (Training/Test)
    train_dir = os.path.join(base_dir, "train")
    test_dir = os.path.join(base_dir, "test")

    if not os.path.exists(train_dir):
        train_dir = os.path.join(base_dir, "Training")
        test_dir = os.path.join(base_dir, "Test")

    print(f"✅ Ruta localizada: {base_dir}")
    print(f"Contenido de {base_dir}: {os.listdir(base_dir)}")
else:
    raise FileNotFoundError("❌ ERROR: No se encontró la carpeta de datos. Revisa la estructura del repo.")

✅ Ruta localizada: /content/Clasificaciondefrutas.udea.novateam/data/selected
Contenido de /content/Clasificaciondefrutas.udea.novateam/data/selected: ['test', 'train']


In [5]:
# ========================================================
# 3. GENERADORES Y CARGA DE DATOS
# ========================================================
img_size = (128, 128)
batch_size = 32
val_split = 0.2

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=val_split)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size,
    class_mode='categorical', subset='training', shuffle=True
)

val_data = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size,
    class_mode='categorical', subset='validation', shuffle=True
)

test_data = test_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size,
    class_mode='categorical', shuffle=False
)

Found 3914 images belonging to 10 classes.
Found 975 images belonging to 10 classes.
Found 1638 images belonging to 10 classes.


In [6]:
# ========================================================
# 4. DEFINICIÓN Y COMPILACIÓN DEL MODELO
# ========================================================
num_classes = len(train_data.class_indices)

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
# ========================================================
# 5. ENTRENAMIENTO
# ========================================================
epochs = 15
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.8195 - loss: 1.4755 - val_accuracy: 0.1005 - val_loss: 61.3485
Epoch 2/15
 73/123 ━━━━━━━━━━━━━━━━━━━━ 1:18 2s/step - accuracy: 0.9766 - loss: 0.4527

KeyboardInterrupt: 

In [ ]:
# ========================================================
# 6. VISUALIZACIÓN DE RESULTADOS (Tu parte faltante)
# ========================================================
plt.figure(figsize=(12, 5))

# Gráfica de Precisión
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Precisión Entreno')
plt.plot(history.history['val_accuracy'], label='Precisión Val')
plt.title('Curva de Precisión')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)

# Gráfica de Pérdida
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Pérdida Entreno')
plt.plot(history.history['val_loss'], label='Pérdida Val')
plt.title('Curva de Pérdida')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ========================================================
# 7. MATRIZ DE CONFUSIÓN Y TABLA FINAL
# ========================================================
test_labels = test_data.classes
predictions = model.predict(test_data)
predicted_classes = np.argmax(predictions, axis=1)

cm = confusion_matrix(test_labels, predicted_classes)
class_names = list(train_data.class_indices.keys())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicho')
plt.ylabel('Verdadero')
plt.show()

# Tabla consolidada
df_resultados = pd.DataFrame(history.history)
df_resultados.insert(0, 'Época', range(1, len(df_resultados) + 1))
display(df_resultados)